In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
    # for filename in filenames:
        # print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Importing Required Libraries

In [2]:
# Visualization 
import matplotlib.pyplot as plt
import seaborn as sns

# Audio specific packages
import librosa
from IPython.display import Audio, display

import torch
from torch import nn
from torch.utils.data import Dataset
import torchaudio

from glob import glob
import random
from tqdm import tqdm
import soundfile as sf
from joblib import Parallel, delayed

# EXPLORATORY DATA ANALYSIS

**The dataset structure we are working with is:**


                messy_mashup
                │
                ├── ESC-50-master
                │    │
                │    ├── audio (2000 Samples spanning 50 environmental noises)
                │    ├── meta
                │    │   │
                │    │   ├── esc50-human.xlsx (Classification report form some model useless for our task)
                │    │   └── esc50.csv (Label for all 2000 audio samples)
                │    │
                │    ├── LICENSE
                │    └── README.md
                │           
                ├── genres_stems (100 examples per genre)
                ├── mashups (3020 test audio files)
                ├── sample_submission.csv
                └── test.csv (id, location)

                

There are equal number of examples for each category to train so there is no class imbalance.

In [4]:
# noise data

noise_df = pd.read_csv('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta/esc50.csv')
noise_df[:5]

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


All 50 categories have an equal number of audio examples, specifically 40 each.

The columns esc10, src_file, and take are not relevant to our analysis.

In [3]:
noise_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/"

In [4]:
# function to plot audio graph and display audio
def audio_and_plot(path , title):
    
    audio_wave,sampling_rate = librosa.load(path)
    
    audio_wave , _ = librosa.effects.trim(audio_wave) # using trim to remove silent parts
    
    display(Audio(path))   # displaying original audio

    # frequency over time
    audio_wave_Series = pd.Series(audio_wave)
    audio_wave_Series.plot(figsize=(14,5) , lw=0.5 , title=title)  # ploting trimmed audio
    plt.show()

    # mel spectrogram
    mel_spec = librosa.feature.melspectrogram(y=audio_wave, sr=sampling_rate, n_mels=128)
    
    #Convert to Decibels 
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    # 5. Plot Mel Spectrogram
    plt.figure(figsize=(14, 5))
    librosa.display.specshow(mel_spec_db, 
                             sr=sampling_rate, 
                             x_axis='time', 
                             y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f"Mel Spectrogram: {title}")
    plt.tight_layout()
    plt.show()


In [ ]:
# some example noise's
print(f"Category: {noise_df['category'][0]}")
audio_and_plot(noise_path +noise_df['filename'][0] , noise_df['category'][0])

print(f"Category: {noise_df['category'][1]}")
audio_and_plot(noise_path+noise_df['filename'][1] , noise_df['category'][1])

print(f"Category: {noise_df['category'][2]}")
audio_and_plot(noise_path+noise_df['filename'][2] , noise_df['category'][2])

The **genres_stems** folder now contains 10 subfolders, one for each genre.

Training data in **genres_stems** consists of four separate stems: *bass.wav*, *drums.wav*, *other.wav*, and *vocals.wav*.

The **test (mashup)** data consists of single, mixed .wav files. 

We must standardize the format of either the test or training data to ensure consistency across both datasets


Standardization of training and testing data can be done in these two ways:

1. Decomposition: **test (mashup)** data into four separate stems.
2. Composition (selected): Merging individual training stems into a single "mashup" track.

We utilize Option 2. This approach allows for the creation of an exponentially larger training set using same ideas of merging the data as provided in data description for **mashups**.

* Selecting instrument stems from different songs of the same genre
* Applying tempo adjustments where needed to ensure rhythmic synchronization
* Mixing the synchronized stems into a single audio track
* Adding one or more random noise samples from the ESC-50 dataset at random positions and intensities



# Creating Training data from genres_stems

In [5]:
genres = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

data_size = 10000            # per genre
target_duration = 30         # 30 seconds
sampling_rate = 22050                   # sampling_rate
num_samples = target_duration * sampling_rate

In [8]:
# pre loading noise files
 
noise_files = list(glob(f'{noise_path}*.wav'))
loaded_noises = []

for i, p in tqdm(enumerate(noise_files)):    
    y, _ = librosa.load(p, sr=sampling_rate) 
    loaded_noises.append(y)

print(f"Loaded {len(loaded_noises)} noise files into RAM.")

2000it [00:14, 137.39it/s]

Loaded 2000 noise files into RAM.


In [ ]:
# creating mashup data to train model

outpit_dir = "generated_mashups"
os.makedirs(outpit_dir, exist_ok=True)

cache = {}

def generating_training_data(item):    # item is genre
    # grouping all four parts of data to make diverse training data
    base_path = f'/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/{item}/*/'
    
    genres_bass = list(glob(base_path + "bass.wav"))
    genres_drums = list(glob(base_path + "drums.wav")) 
    genres_other = list(glob(base_path + "other.wav"))
    genres_vocals = list(glob(base_path + "vocals.wav"))

    genre_folder = os.path.join(outpit_dir, f'train_{item}')
    os.makedirs(genre_folder, exist_ok=True)
    
    for i in tqdm(range(data_size)):
        stem_paths = [
            random.choice(genres_drums),
            random.choice(genres_bass),
            random.choice(genres_other),
            random.choice(genres_vocals)
                     ]

        sampling_rate = 22050  
        stem_audio_signals = []
        bpm = None

        # merging stems
        
        for path in stem_paths:
            
            audio , sr = librosa.load(path, sr=sampling_rate)  # loading the audio
            
            if path not in cache:  
                onset_env = librosa.onset.onset_strength(y=audio, sr=sampling_rate)
                current_bpm = librosa.feature.tempo(onset_envelope=onset_env)[0]
            
                cache[path] = current_bpm  # caching compute heavy tasks
            else:
                current_bpm = cache[path]
                
            
            if current_bpm <= 0:            #handle silence
                current_bpm = 120.0

            if bpm:       # for bass, vocals and other
                rate = bpm / current_bpm     # to match the bpm of drums
                audio = librosa.effects.time_stretch(audio, rate=rate)
                        

            else:         # for drums to decide the bpm 
                bpm = current_bpm
                
            y_audio = librosa.util.fix_length(audio, size=num_samples)
            stem_audio_signals.append(y_audio)

        mixed_signal = np.sum(stem_audio_signals, axis=0) / 4.0


        # adding noise 
        noise_to_add = np.random.randint(0,3)  # 3 noise = 15sec of noise in 30sec of audio
        
        for _ in range(noise_to_add):
            
            # noise_p = random.choice(noise)
            # noise_a, _ = librosa.load(noise_p, sr=sampling_rate)
            noise_a = random.choice(loaded_noises)
                
            noise_container = np.zeros(len(mixed_signal))

            start_pos = np.random.randint(0,len(mixed_signal)-(len(noise_a)))
            noise_container[start_pos:start_pos+len(noise_a)] = noise_a
            
            noise_intensity = random.uniform(0.1, 1)    # add more randomness to noises
            mixed_signal = mixed_signal + (noise_container * noise_intensity)

        filename = f'audio{i}.wav'
        full_save_path = os.path.join(genre_folder, filename)
        sf.write(full_save_path, mixed_signal, sampling_rate)

    # audio_and_plot(f"{genre_folder}/audio1.wav" ,item)
    # audio_and_plot(f"{genre_folder}/audio2.wav" ,item)


# executing genres parallelly
Parallel(n_jobs=-1)(
    delayed(generating_training_data)(g) 
    for g in genres
)

  1%|          | 60/10000 [02:26<4:37:12,  1.67s/it] 

In [ ]:
import shutil
shutil.make_archive(zip_name, 'zip', root_dir='.', base_dir=output_dir)